# Ghost Pharmacies

When Bay Area pharmacies closed, did people in cheaper homes lose their nearest pharmacy more often?

This one notebook goes from the raw files in `data/` to every number, chart and map we report.
**Rule:** before submitting, do *Runtime → Restart and run all* (or `Kernel → Restart & Run All`) and make sure it finishes without errors.

Build steps follow the Team Master Guide. Write your name next to the step you own.

## Step 0. Setup
Runs in both Google Colab and a local Jupyter. Edit only the paths in this cell.

In [3]:
import os, sys, random
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "folium"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    # Shared Drive folder that holds the raw files and receives outputs. Change if yours differs.
    PROJECT_DIR = Path("/content/drive/MyDrive/ghost-pharmacies")
else:
    # Local: the repository root (this notebook lives in notebooks/).
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_DIR / "data"        # raw files, never edited
OUT_DIR = PROJECT_DIR / "outputs"      # everything the notebook creates
FIG_DIR = OUT_DIR / "figures"
MAP_DIR = OUT_DIR / "maps"
for d in (OUT_DIR, FIG_DIR, MAP_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Raw file names. Rename here if the event files are named differently.
PLACES_CSV = DATA_DIR / "google_maps.csv"
ZILLOW_CSVS = [DATA_DIR / n for n in ["zillow_part1.csv", "zillow_part2.csv", "zillow_part3.csv"]]   # full Zillow file, in parts: read all and concatenate
CENSUS_CSV = DATA_DIR / "census_tracts.csv"   # optional

# One seed, used everywhere (train/test split, sampling, models).
SEED = 42
random.seed(SEED)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import folium

np.random.seed(SEED)
pd.set_option("display.max_columns", 50)

print("Colab" if IN_COLAB else "Local", "| project dir:", PROJECT_DIR)
print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__,
      "| sklearn", sklearn.__version__, "| folium", folium.__version__)

Local | project dir: /Users/brianlee/ghost-pharmacies
python 3.14.5 | pandas 3.1.1 | numpy 2.4.6 | sklearn 1.9.1 | folium 0.20.0


### Check the raw files
Confirms each file is in place and has the columns the guide relies on. Reads only the header row, so it is fast even on the 500 MB file.

In [4]:
ZILLOW_COLS = ["zpid", "latitude", "longitude", "county", "bedrooms", "homeStatus", "price",
               "rentZestimate", "getting_around_scores"]
EXPECTED = {
    PLACES_CSV: ["place_id", "name", "category", "lat", "lon", "permanently_closed", "top_reviews"],
    **{p: ZILLOW_COLS for p in ZILLOW_CSVS},
}

for path, cols in EXPECTED.items():
    if not path.exists():
        print(f"MISSING  {path.name}: put it in {DATA_DIR}")
        continue
    header = pd.read_csv(path, nrows=0).columns
    missing = [c for c in cols if c not in header]
    size_mb = path.stat().st_size / 1e6
    status = "OK      " if not missing else "CHECK   "
    print(f"{status} {path.name} ({size_mb:,.0f} MB, {len(header)} columns)"
          + (f" missing: {missing}" if missing else ""))

print("Census (optional):", "found" if CENSUS_CSV.exists() else "not added")

OK       google_maps.csv (506 MB, 41 columns)
OK       zillow_part1.csv (1,495 MB, 140 columns)
OK       zillow_part2.csv (3,934 MB, 140 columns)
OK       zillow_part3.csv (2,006 MB, 140 columns)
Census (optional): not added


## Step 1. Build the pharmacy table
Owner: ___  
Load Places with `usecols`, keep `category == 'Pharmacy'` (expect about 1,694), add the `chain` column, print closures per chain and compare with the guide.

In [5]:
# TODO

## Step 2. Decide which closures are real
Owner: ___  
Distance from each closed pharmacy to the nearest open one (BallTree, haversine). Real closure = not Rite Aid or Lucky and no open pharmacy within 50 m (expect about 358). Save `outputs/pharmacies_clean.csv`.

In [6]:
# TODO

## Step 3. Hand-check
Owner: ___  
30 real closures + 10 open pharmacies sampled with `SEED`, checked on Google Maps. Load the results table here and report the error rate.

In [7]:
# TODO

## Step 4. Build the rent level
Owner: ___  
Load Zillow (use `chunksize` if needed), drop missing coordinates, compute rent and relative rent by bedrooms, keep \$500 to \$20,000 and 0 to 5 bedrooms, split into thirds within county.

In [8]:
# 4.1: processing the 3 zillow files into 1 table
pieces = []
for i in range(len(ZILLOW_CSVS)):
    reader = pd.read_csv(ZILLOW_CSVS[i], usecols=ZILLOW_COLS, chunksize=50000)
    for chunk in reader:
        chunk["row_source"] = i + 1 # tagging each row for easy reference to its file origin
        pieces.append(chunk)
mergedTable = pd.concat(pieces, ignore_index=True)

In [9]:
# confirmation of 4.1 implementation
display(mergedTable.head(10))
print(len(mergedTable))
print(mergedTable.dtypes)
print(mergedTable["zpid"].duplicated().sum())
print(mergedTable["row_source"].value_counts())

,zpid,homeStatus,bedrooms,price,longitude,latitude,rentZestimate,county,getting_around_scores,row_source
0,2073071533,FOR_RENT,2.0,5094,-122.263677,37.510161,NaN,San Mateo County,"{""bike_score"":""66/100 | Bikeable"",""transit_sco...",1
1,2077933815,FOR_RENT,3.0,5614,-122.025500,37.376560,NaN,Santa Clara County,"{""bike_score"":""84/100 | Very Bikeable"",""transi...",1
2,19253238,RECENTLY_SOLD,4.0,1850000,-122.532330,37.922455,NaN,Marin County,"{""bike_score"":""43/100 | Somewhat Bikeable"",""tr...",1
3,326139325,RECENTLY_SOLD,1.0,749000,-122.456100,37.764890,5221.0,San Francisco County,"{""bike_score"":""67/100 | Bikeable"",""transit_sco...",1
4,2089306902,FOR_RENT,3.0,5819,-121.932376,37.347085,NaN,Santa Clara County,"{""bike_score"":""79/100 | Very Bikeable"",""transi...",1
5,2067771850,FOR_RENT,0.0,2095,-122.415020,37.788602,NaN,San Francisco County,"{""bike_score"":""70/100 | Very Bikeable"",""transi...",1
6,19278440,RECENTLY_SOLD,3.0,679000,-122.570625,38.117126,3929.0,Marin County,"{""bike_score"":""29/100 | Somewhat Bikeable"",""tr...",1
7,299071045,RECENTLY_SOLD,4.0,840000,-122.024370,37.596542,2963.0,Alameda County,"{""bike_score"":""59/100 | Bikeable"",""transit_sco...",1
8,18402887,RECENTLY_SOLD,4.0,1150000,-122.088776,37.949580,5115.0,Contra Costa County,"{""bike_score"":""51/100 | Bikeable"",""transit_sco...",1
9,18384685,RECENTLY_SOLD,4.0,1452000,-122.036670,37.931816,4582.0,Contra Costa County,"{""bike_score"":""86/100 | Very Bikeable"",""transi...",1


148005
zpid                       int64
homeStatus                   str
bedrooms                 float64
price                      int64
longitude                float64
latitude                 float64
rentZestimate            float64
county                       str
getting_around_scores        str
row_source                 int64
dtype: object
0
row_source
2    78424
3    39867
1    29714
Name: count, dtype: int64


In [10]:
# 4.2: visualizing data before changing anything
print(mergedTable["homeStatus"].value_counts())
print(mergedTable["county"].value_counts())
print(mergedTable.isna().mean())
print(mergedTable.groupby("homeStatus")["price"].median())

homeStatus
FOR_RENT         74343
RECENTLY_SOLD    61606
FOR_SALE         12056
Name: count, dtype: int64
county
Santa Clara County      38559
Alameda County          34588
Contra Costa County     19512
San Francisco County    15652
San Mateo County        13702
Sonoma County           10216
Solano County            8442
Marin County             4474
Napa County              2860
Name: count, dtype: int64
zpid                     0.000000
homeStatus               0.000000
bedrooms                 0.015452
price                    0.000000
longitude                0.005371
latitude                 0.005371
rentZestimate            0.513354
county                   0.000000
getting_around_scores    0.000000
row_source               0.000000
dtype: float64
homeStatus
FOR_RENT            3109.0
FOR_SALE          949000.0
RECENTLY_SOLD    1103500.0
Name: price, dtype: float64


In [11]:
# 4.3: Clean data, drop homes w no locations
homes = mergedTable.dropna(subset=["longitude", "latitude"])
print("rows after:", len(homes))


rows after: 147210


In [12]:
# 4.4: compute rent
homes["rent"] = np.where(homes["homeStatus"] == "FOR_RENT", homes["price"], homes["rentZestimate"])
print(homes["rent"].isna().groupby(homes["homeStatus"]).mean())

# 4.5: filter out noise
# keep contains T/F list that lines up with the rows of homes, rows that line up with F are dropped from homes
keep = homes["rent"].between(500, 20000) & homes["bedrooms"].between(0, 5)
homes = homes[keep]
# sanity check
print("rows after 4.5:", len(homes))
homes["homeStatus"].value_counts()


homeStatus
FOR_RENT         0.000000
FOR_SALE         0.302618
RECENTLY_SOLD    0.086632
Name: rent, dtype: float64
rows after 4.5: 134835


homeStatus
FOR_RENT         73519
RECENTLY_SOLD    53588
FOR_SALE          7728
Name: count, dtype: int64

In [ ]:
# 4.6: relative rent price so we can more accurately map what is a "cheap home"
# groupby to sort rows into groups by # beds, specified that the rent values are the values in the groups, then got the median
homes["rel_rent"] = homes["rent"] / homes.groupby("bedrooms")["rent"].transform("median")
display(homes.head(10))
# sanity check
print(homes.groupby("bedrooms")["rent"].agg(["count", "median"]))


# 4.7: label each homes rel rent by low, middle, high comparative to county so we can analyze impact of closing pharmas
prcnt_by_row = homes.groupby("county")["rel_rent"].rank(pct=True, method="first") # rank gives percentile of the rel_rent relative to the group
homes["rent_tier"] = pd.cut(prcnt_by_row, bins=[0, 1/3, 2/3, 1], labels=["low", "middle", "high"]) # sorts the prcnts into the labels and makes a new col
display(homes.head(10))
# sanity check
print(len(prcnt_by_row) == len(homes))
print(pd.crosstab(homes["county"], homes["rent_tier"]))
print(homes["rent_tier"].isna().sum())

,zpid,homeStatus,bedrooms,price,longitude,latitude,rentZestimate,county,getting_around_scores,row_source,rent,rel_rent,rent_tier
0,2073071533,FOR_RENT,2.0,5094,-122.263677,37.510161,NaN,San Mateo County,"{""bike_score"":""66/100 | Bikeable"",""transit_sco...",1,5094.0,1.482106,high
1,2077933815,FOR_RENT,3.0,5614,-122.025500,37.376560,NaN,Santa Clara County,"{""bike_score"":""84/100 | Very Bikeable"",""transi...",1,5614.0,1.337622,high
3,326139325,RECENTLY_SOLD,1.0,749000,-122.456100,37.764890,5221.0,San Francisco County,"{""bike_score"":""67/100 | Bikeable"",""transit_sco...",1,5221.0,1.802831,high
4,2089306902,FOR_RENT,3.0,5819,-121.932376,37.347085,NaN,Santa Clara County,"{""bike_score"":""79/100 | Very Bikeable"",""transi...",1,5819.0,1.386467,high
5,2067771850,FOR_RENT,0.0,2095,-122.415020,37.788602,NaN,San Francisco County,"{""bike_score"":""70/100 | Very Bikeable"",""transi...",1,2095.0,0.859664,low
6,19278440,RECENTLY_SOLD,3.0,679000,-122.570625,38.117126,3929.0,Marin County,"{""bike_score"":""29/100 | Somewhat Bikeable"",""tr...",1,3929.0,0.936145,low
7,299071045,RECENTLY_SOLD,4.0,840000,-122.024370,37.596542,2963.0,Alameda County,"{""bike_score"":""59/100 | Bikeable"",""transit_sco...",1,2963.0,0.602358,low
8,18402887,RECENTLY_SOLD,4.0,1150000,-122.088776,37.949580,5115.0,Contra Costa County,"{""bike_score"":""51/100 | Bikeable"",""transit_sco...",1,5115.0,1.039845,high
9,18384685,RECENTLY_SOLD,4.0,1452000,-122.036670,37.931816,4582.0,Contra Costa County,"{""bike_score"":""86/100 | Very Bikeable"",""transi...",1,4582.0,0.931490,high
11,2055832490,FOR_RENT,1.0,2195,-122.098081,37.395047,NaN,Santa Clara County,"{""bike_score"":null,""transit_score"":""45/100 | S...",1,2195.0,0.757942,low


          count  median
bedrooms               
0.0        7876  2437.0
1.0       34744  2896.0
2.0       42133  3437.0
3.0       28442  4197.0
4.0       17027  4919.0
5.0        4613  5747.0


,zpid,homeStatus,bedrooms,price,longitude,latitude,rentZestimate,county,getting_around_scores,row_source,rent,rel_rent,rent_tier
0,2073071533,FOR_RENT,2.0,5094,-122.263677,37.510161,NaN,San Mateo County,"{""bike_score"":""66/100 | Bikeable"",""transit_sco...",1,5094.0,1.482106,high
1,2077933815,FOR_RENT,3.0,5614,-122.025500,37.376560,NaN,Santa Clara County,"{""bike_score"":""84/100 | Very Bikeable"",""transi...",1,5614.0,1.337622,high
3,326139325,RECENTLY_SOLD,1.0,749000,-122.456100,37.764890,5221.0,San Francisco County,"{""bike_score"":""67/100 | Bikeable"",""transit_sco...",1,5221.0,1.802831,high
4,2089306902,FOR_RENT,3.0,5819,-121.932376,37.347085,NaN,Santa Clara County,"{""bike_score"":""79/100 | Very Bikeable"",""transi...",1,5819.0,1.386467,high
5,2067771850,FOR_RENT,0.0,2095,-122.415020,37.788602,NaN,San Francisco County,"{""bike_score"":""70/100 | Very Bikeable"",""transi...",1,2095.0,0.859664,low
6,19278440,RECENTLY_SOLD,3.0,679000,-122.570625,38.117126,3929.0,Marin County,"{""bike_score"":""29/100 | Somewhat Bikeable"",""tr...",1,3929.0,0.936145,low
7,299071045,RECENTLY_SOLD,4.0,840000,-122.024370,37.596542,2963.0,Alameda County,"{""bike_score"":""59/100 | Bikeable"",""transit_sco...",1,2963.0,0.602358,low
8,18402887,RECENTLY_SOLD,4.0,1150000,-122.088776,37.949580,5115.0,Contra Costa County,"{""bike_score"":""51/100 | Bikeable"",""transit_sco...",1,5115.0,1.039845,high
9,18384685,RECENTLY_SOLD,4.0,1452000,-122.036670,37.931816,4582.0,Contra Costa County,"{""bike_score"":""86/100 | Very Bikeable"",""transi...",1,4582.0,0.931490,high
11,2055832490,FOR_RENT,1.0,2195,-122.098081,37.395047,NaN,Santa Clara County,"{""bike_score"":null,""transit_score"":""45/100 | S...",1,2195.0,0.757942,low


True
rent_tier               low  middle   high
county                                    
Alameda County        10869   10870  10870
Contra Costa County    6079    6079   6080
Marin County           1255    1255   1256
Napa County             788     788    788
San Francisco County   4541    4542   4542
San Mateo County       3928    3928   3928
Santa Clara County    11866   11867  11867
Solano County          2629    2629   2630
Sonoma County          2987    2987   2987
0


In [ ]:
# 4.8: create col for numeric walk score values, if none put NaN
pattern = r'"walk_score":"(\d+)/100'
homes["walk_score"] = homes["getting_around_scores"].str.extract(pattern, expand=False).astype(float) # extractin regex pattern as a collection, turned into floats, and added to new col
display(homes.sample(n=10)
)
# sanity check
print(homes["walk_score"].describe(), "\n")
print("percent with walk score:", homes["walk_score"].notna().mean())

,zpid,homeStatus,bedrooms,price,longitude,latitude,rentZestimate,county,getting_around_scores,row_source,rent,rel_rent,rent_tier,walk_score
111878,18332228,RECENTLY_SOLD,3.0,422750,-121.920780,38.026430,2472.0,Contra Costa County,"{""bike_score"":""35/100 | Somewhat Bikeable"",""tr...",3,2472.0,0.588992,low,34.0
117905,2055102964,FOR_RENT,2.0,2950,-122.311980,37.903618,2309.0,Contra Costa County,"{""bike_score"":""64/100 | Bikeable"",""transit_sco...",3,2950.0,0.858307,middle,77.0
4892,444342507,FOR_RENT,2.0,1949,-121.969131,38.355032,NaN,Solano County,"{""bike_score"":""44/100 | Somewhat Bikeable"",""tr...",1,1949.0,0.567064,low,57.0
90231,461570667,FOR_RENT,1.0,4845,-122.392764,37.787390,NaN,San Francisco County,"{""bike_score"":""57/100 | Bikeable"",""transit_sco...",2,4845.0,1.672997,middle,95.0
38729,353284432,FOR_RENT,1.0,3709,-122.263909,37.555912,NaN,San Mateo County,"{""bike_score"":""55/100 | Bikeable"",""transit_sco...",2,3709.0,1.280732,middle,70.0
65588,2069511820,FOR_RENT,0.0,1995,-122.259955,37.858784,NaN,Alameda County,"{""bike_score"":""98/100 | Biker's Paradise"",""tra...",2,1995.0,0.818629,low,91.0
39351,24966653,RECENTLY_SOLD,4.0,1175000,-122.056850,37.688038,4121.0,Alameda County,"{""bike_score"":""22/100 | Somewhat Bikeable"",""tr...",2,4121.0,0.837772,middle,22.0
107440,19792605,RECENTLY_SOLD,4.0,1935000,-121.768560,37.328217,5323.0,Santa Clara County,"{""bike_score"":""22/100 | Somewhat Bikeable"",""tr...",2,5323.0,1.082131,middle,2.0
1845,303806825,RECENTLY_SOLD,2.0,1600000,-122.468240,37.763270,6178.0,San Francisco County,"{""bike_score"":""85/100 | Very Bikeable"",""transi...",1,6178.0,1.797498,high,99.0
91477,15145378,RECENTLY_SOLD,3.0,3855000,-122.431470,37.757770,7690.0,San Francisco County,"{""bike_score"":""60/100 | Bikeable"",""transit_sco...",2,7690.0,1.832261,high,95.0


count    114032.000000
mean         59.872343
std          28.030395
min           0.000000
25%          40.000000
50%          64.000000
75%          83.000000
max         100.000000
Name: walk_score, dtype: float64 

percent with walk score: 0.8457151333110839


In [ ]:
# 4.8 check: 15% no walk score? is that a pattern mess up or truly missing data

miss = homes.loc[homes["walk_score"].isna(), "getting_around_scores"]

n_missing     = len(miss)
n_empty_col   = miss.isna().sum()                                                # whole column empty
n_null_walk   = miss.str.contains('"walk_score":null', regex=False).sum()        # walk score explicitly null
n_unexplained = n_missing - n_empty_col - n_null_walk

print("homes with no walk_score:     ", n_missing)
print("  getting_around_scores empty:", n_empty_col)
print("  walk_score is null in text: ", n_null_walk)
print("  unexplained (possible miss):", n_unexplained)
print("dtype:", homes["walk_score"].dtype)

# Show a few unexplained rows in full so we can see how they're written
unexplained = miss[miss.notna() & ~miss.str.contains('"walk_score":null', regex=False)]
for s in unexplained.head(3):
    print("\n", s)

homes with no walk_score:      20803
  getting_around_scores empty: 0
  walk_score is null in text:  20803
  unexplained (possible miss): 0
dtype: float64


## Step 5. Access-loss analysis
Owner: ___  
Nearest-pharmacy distance before and after for every home, the two-sided table, with and without chain shutdowns, broken down by county.

In [ ]:
# TODO

## Step 6. Split the data and train the first model
Owner: ___  
80/20 split stratified on the label with `random_state=SEED`. Save test IDs to `outputs/test_ids.csv`. **Do not look at test scores until Step 8.** Baseline: logistic regression on chain only. Model: gradient boosting. Compare with cross-validation on the training set.

In [ ]:
# TODO

## Step 7. Improve the model, carefully
Owner: ___  
Two or three changes, judged by cross-validation on training only. No `rating`, `reviews_count`, `open_hours` or `is_claimed` (they leak). Freeze the model at the end of this step.

In [ ]:
# TODO

## Step 8. Final evaluation and fairness
Owner: ___  
Score model and baseline on the test set once. AUC, plus precision and recall for the top 20%. Subgroups: low, middle, high rent; chain vs independent. Five failure examples.

In [ ]:
# TODO

## Step 9. Map and charts
Owner: ___  
Save charts to `FIG_DIR` and the folium map to `MAP_DIR`. Two-sided bars, 800 m before/after, model vs baseline, "last pharmacy standing" list.

In [ ]:
# TODO

## Step 10. Package
Owner: ___  
Restart and run all from the raw files. Update the README.

In [ ]:
# TODO